# LeetCode 2144 — Minimum Cost of Buying Candies With Discount bằng Python

Link đề: https://leetcode.com/problems/minimum-cost-of-buying-candies-with-discount/ · Độ khó: **Easy** · Chủ đề: Array, Greedy, Sorting

```python
class Solution:
    def minimumCost(self, cost: List[int]) -> int:

```

## 1. Mục tiêu học

Sau bài này bạn sẽ:

1. Hiểu đề: mua 2 viên kẹo bất kỳ thì được tặng 1 viên có giá **không vượt quá** giá rẻ hơn trong hai viên đã mua.
2. Làm quen với **tham lam (greedy)**: sắp xếp giảm dần rồi cứ mỗi 3 viên thì viên thứ 3 được miễn phí.
3. Dùng `sorted(..., reverse=True)`, lát cắt có bước `a[2::3]`, `sum`, và `heapq` (heap).
4. So sánh 4 cách: mô phỏng brute force, sort + vòng lặp, sort + lát cắt, heap.
5. Hiểu vì sao greedy đúng (lập luận hoán đổi), trace, test ngẫu nhiên đối chiếu với brute force và tránh các lỗi hay gặp (sort tăng dần, bỏ sót nhóm cuối chưa đủ 3).

## 2. Đề bài & ví dụ

Một cửa hàng giảm giá kẹo: **mua 2 viên kẹo bất kỳ thì được tặng thêm 1 viên** miễn phí, với điều kiện giá viên được tặng **nhỏ hơn hoặc bằng giá thấp hơn** trong hai viên đã mua.

Cho `cost[i]` là giá viên kẹo `i`. Hãy trả về **tổng chi phí tối thiểu** để lấy **tất cả** các viên kẹo.

| Ví dụ | cost | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,2,3]` | `5` | mua 2 và 3 (trả 5), tặng 1 |
| 2 | `[6,5,7,9,2,2]` | `23` | mua 9,7 tặng 6; mua 5,2 tặng 2; trả 9+7+5+2 = 23 |
| 3 | `[5,5]` | `10` | chỉ có 2 viên, không có khuyến mãi |

Constraints: `1 <= cost.length <= 100`, `1 <= cost[i] <= 100`.

Lưu ý: nếu chỉ còn 1 hoặc 2 viên chưa nhận thì phải trả đủ tiền (không thể nhận quà khi chưa mua đủ 2).

In [1]:
cost = [6, 5, 7, 9, 2, 2]
print("Tổng giá nếu trả đủ:", sum(cost))
print("Giảm giá tối đa là bao nhiêu để ra 23?", sum(cost) - 23)

Tổng giá nếu trả đủ: 31
Giảm giá tối đa là bao nhiêu để ra 23? 8


## 3. Kiến thức nền

### 3.1. `sorted(a, reverse=True)` — sắp xếp giảm dần

`sorted` trả về **list mới** đã sắp xếp, không đổi `a`. `reverse=True` cho thứ tự giảm dần. `a.sort()` thì sắp xếp **tại chỗ** và trả `None`.

In [2]:
a = [6, 5, 7, 9, 2, 2]
print(sorted(a, reverse=True))
print(a)                       # a không bị đổi
b = a[:]                       # copy
b.sort(reverse=True)
print(b)

[9, 7, 6, 5, 2, 2]
[6, 5, 7, 9, 2, 2]
[9, 7, 6, 5, 2, 2]


### 3.2. Lát cắt có bước: `a[start::step]`

`a[2::3]` lấy phần tử ở chỉ số `2, 5, 8, ...` (mỗi phần tử thứ 3 bắt đầu từ chỉ số 2). Đây chính là các viên **được tặng** khi đã sắp xếp giảm dần.

In [3]:
a = [9, 7, 6, 5, 2, 2]
print(a[2::3])    # [6, 2]
print(a[0::3])    # [9, 5]
print(a[1::3])    # [7, 2]

[6, 2]
[9, 5]
[7, 2]


### 3.3. Phép chia lấy dư `%` để tuần hoàn theo nhóm 3

`i % 3` chạy `0, 1, 2, 0, 1, 2, ...`. Vị trí có `i % 3 == 2` là viên thứ ba của mỗi nhóm.

In [4]:
for i in range(7):
    print(i, i % 3, "<- tặng" if i % 3 == 2 else "")

0 0 
1 1 
2 2 <- tặng
3 0 
4 1 
5 2 <- tặng
6 0 


### 3.4. `heapq` — hàng đợi ưu tiên

Python chỉ có **min-heap**: `heapq.heappop` lấy ra phần tử **nhỏ nhất**. Để lấy lớn nhất ta đẩy số **âm** (`-x`) vào heap. Mỗi lần push/pop tốn `O(log n)`.

In [5]:
import heapq

h = []
for x in [6, 5, 7, 9]:
    heapq.heappush(h, -x)
print([-heapq.heappop(h) for _ in range(len(h))])   # [9, 7, 6, 5]

[9, 7, 6, 5]


## 4. Ý tưởng nền tảng — tính bằng tay

Ví dụ 2: `cost = [6,5,7,9,2,2]`. Ta muốn **miễn phí viên đắt nhất có thể**, nhưng viên được tặng không đắt hơn viên rẻ hơn trong cặp mua.

- Mua hai viên đắt nhất: `9` và `7` -> được tặng viên lớn nhất còn lại không vượt `7`: đó là `6`.
- Còn `5, 2, 2`: mua `5` và `2` -> tặng `2`.

Trả `9 + 7 + 5 + 2 = 23`. Ý chính: **luôn mua hai viên đắt nhất, nhận viên đắt thứ ba**. Làm đi làm lại cho đến hết.

In [6]:
s = sorted(cost, reverse=True)
print(s)
print("Nhóm 1:", s[0:3], "mua", s[0:2], "tặng", s[2])
print("Nhóm 2:", s[3:6], "mua", s[3:5], "tặng", s[5])
print("Chi phí:", s[0] + s[1] + s[3] + s[4])

[9, 7, 6, 5, 2, 2]
Nhóm 1: [9, 7, 6] mua [9, 7] tặng 6
Nhóm 2: [5, 2, 2] mua [5, 2] tặng 2
Chi phí: 23


## 5. Cách 1 — Brute Force: mô phỏng "lấy hai đắt nhất, tặng đắt thứ ba"

Dùng một list `rest` các viên chưa lấy. Lặp đến khi hết:

1. Tìm viên đắt nhất, trả tiền, xóa nó khỏi `rest`.
2. Nếu còn, tìm viên đắt nhất tiếp theo, trả tiền, xóa.
3. Nếu còn, tìm viên đắt nhất tiếp theo, **miễn phí**, xóa.

Mỗi lần "tìm đắt nhất" là `max(rest)` -> `O(n)`; `rest.remove(x)` cũng `O(n)`.

In [7]:
def minimum_cost_brute(cost):
    rest = list(cost)
    total = 0
    while rest:
        for step in range(3):
            if not rest:
                break
            biggest = max(rest)
            rest.remove(biggest)        # xóa một phần tử có giá trị = biggest
            if step < 2:
                total += biggest        # 2 viên đầu phải trả tiền
            # step == 2: viên được tặng, không tính tiền
    return total

print(minimum_cost_brute([1, 2, 3]))               # 5
print(minimum_cost_brute([6, 5, 7, 9, 2, 2]))      # 23
print(minimum_cost_brute([5, 5]))                  # 10

5
23
10


**Độ phức tạp:** `n` vòng, mỗi vòng `max` + `remove` đều `O(n)` -> `O(n²)` thời gian, `O(n)` bộ nhớ (bản sao list). Với `n ≤ 100` rất nhanh nhưng **nút thắt**: ta tìm "lớn nhất" lặp lại nhiều lần trong khi chỉ cần **sắp xếp một lần**.

## 6. Ý tưởng bước đột phá — sắp xếp giảm dần, bỏ viên thứ 3 của mỗi nhóm

Insight: thứ tự lấy kẹo "đắt nhất trước" chính là thứ tự của danh sách **sắp xếp giảm dần**. Nên:

- Sắp xếp giảm dần: `s[0] >= s[1] >= s[2] >= ...`
- Nhóm theo 3: `(s[0], s[1], s[2])`, `(s[3], s[4], s[5])`, ... — trong mỗi nhóm trả cho hai viên đầu, viên thứ ba miễn phí.

Quy tắc: **nếu** chỉ số `i % 3 == 2` **thì** viên đó miễn phí, **ngược lại** trả tiền.

Điều kiện "viên tặng ≤ viên rẻ hơn trong hai viên đã mua" luôn thỏa vì `s[i+2] <= s[i+1]`.

## 7. Các cách tiếp theo

### Cách 2 — Sort giảm dần + vòng lặp với `i % 3`

In [8]:
def minimum_cost_sort_loop(cost):
    s = sorted(cost, reverse=True)
    total = 0
    for i, c in enumerate(s):
        if i % 3 != 2:        # không phải viên thứ ba của nhóm
            total += c
    return total

print(minimum_cost_sort_loop([6, 5, 7, 9, 2, 2]))

23


In [9]:
from typing import List

class SolutionSortLoop:
    def minimumCost(self, cost: List[int]) -> int:
        s = sorted(cost, reverse=True)
        total = 0
        for i, c in enumerate(s):
            if i % 3 != 2:
                total += c
        return total

print(SolutionSortLoop().minimumCost([1, 2, 3]))   # 5

5


Độ phức tạp Cách 2: `O(n log n)` thời gian (sort), `O(n)` bộ nhớ (list đã sort).

### Cách 3 — Tổng trừ đi các viên miễn phí bằng lát cắt `s[2::3]`

`tổng tiền = sum(cost) - sum(s[2::3])`, với `s` đã sắp xếp giảm dần. Rất ngắn.

In [10]:
def minimum_cost_slice(cost):
    s = sorted(cost, reverse=True)
    return sum(s) - sum(s[2::3])

print(minimum_cost_slice([6, 5, 7, 9, 2, 2]), minimum_cost_slice([5, 5]))

23 10


In [11]:
from typing import List

class SolutionSlice:
    def minimumCost(self, cost: List[int]) -> int:
        s = sorted(cost, reverse=True)
        return sum(s) - sum(s[2::3])

print(SolutionSlice().minimumCost([6, 5, 7, 9, 2, 2]))   # 23

23


Độ phức tạp Cách 3: `O(n log n)` thời gian, `O(n)` bộ nhớ.

### Cách 4 — Max-heap bằng `heapq` (số âm)

Đẩy mọi giá vào heap dưới dạng `-giá`. Rồi lấy ra từng viên theo thứ tự giảm dần, đếm `k` để biết viên nào miễn phí. Cách này hữu ích khi dữ liệu đến **dần dần** (stream) hoặc khi chỉ cần xử lý từng viên.

In [12]:
import heapq

def minimum_cost_heap(cost):
    heap = [-c for c in cost]
    heapq.heapify(heap)            # O(n)
    total, k = 0, 0
    while heap:
        c = -heapq.heappop(heap)   # lớn nhất hiện tại
        if k % 3 != 2:
            total += c
        k += 1
    return total

print(minimum_cost_heap([6, 5, 7, 9, 2, 2]))

23


In [13]:
from typing import List
import heapq

class SolutionHeap:
    def minimumCost(self, cost: List[int]) -> int:
        heap = [-c for c in cost]
        heapq.heapify(heap)
        total = k = 0
        while heap:
            c = -heapq.heappop(heap)
            if k % 3 != 2:
                total += c
            k += 1
        return total

print(SolutionHeap().minimumCost([1, 2, 3]))   # 5

5


Độ phức tạp Cách 4: `heapify` `O(n)` + `n` lần `heappop` `O(log n)` = `O(n log n)` thời gian, `O(n)` bộ nhớ.

Ghi chú: vì giá chỉ trong `[1, 100]`, còn có **counting sort** `O(n + 100)`: đếm số lần mỗi giá rồi duyệt từ 100 xuống 1 (xem mục Biến thể).

## 8. TRACE từng bước

Trace Cách 2 trên `cost = [6,5,7,9,2,2]`. Sau khi sắp xếp giảm dần: `s = [9,7,6,5,2,2]`.

| i | c | i % 3 | Hành động | total |
|---|---|---|---|---|
| 0 | 9 | 0 | trả 9 | 9 |
| 1 | 7 | 1 | trả 7 | 16 |
| 2 | 6 | 2 | **tặng** | 16 |
| 3 | 5 | 0 | trả 5 | 21 |
| 4 | 2 | 1 | trả 2 | 23 |
| 5 | 2 | 2 | **tặng** | 23 |

In [14]:
def trace_sort_loop(cost):
    s = sorted(cost, reverse=True)
    print("đã sắp xếp giảm dần:", s)
    total = 0
    for i, c in enumerate(s):
        free = (i % 3 == 2)
        if not free:
            total += c
        print(f"i={i} c={c} i%3={i % 3} -> {'TẶNG' if free else 'trả ' + str(c)}  total={total}")
    return total

print("Kết quả:", trace_sort_loop([6, 5, 7, 9, 2, 2]))

đã sắp xếp giảm dần: [9, 7, 6, 5, 2, 2]
i=0 c=9 i%3=0 -> trả 9  total=9
i=1 c=7 i%3=1 -> trả 7  total=16
i=2 c=6 i%3=2 -> TẶNG  total=16
i=3 c=5 i%3=0 -> trả 5  total=21
i=4 c=2 i%3=1 -> trả 2  total=23
i=5 c=2 i%3=2 -> TẶNG  total=23
Kết quả: 23


In [15]:
def trace_brute(cost):
    rest = list(cost)
    total, round_no = 0, 1
    while rest:
        print(f"-- nhóm {round_no}: còn {sorted(rest, reverse=True)}")
        for step in range(3):
            if not rest:
                break
            b = max(rest)
            rest.remove(b)
            if step < 2:
                total += b
                print(f"   mua {b}, total={total}")
            else:
                print(f"   tặng {b}")
        round_no += 1
    return total

print("Kết quả:", trace_brute([1, 2, 3, 4]))

-- nhóm 1: còn [4, 3, 2, 1]
   mua 4, total=4
   mua 3, total=7
   tặng 2
-- nhóm 2: còn [1]
   mua 1, total=8
Kết quả: 8


## 9. Code nộp LeetCode

Dùng Cách 3 (sort giảm dần, trừ các viên `s[2::3]`).

In [16]:
from typing import List, Optional

class Solution:
    def minimumCost(self, cost: List[int]) -> int:
        s = sorted(cost, reverse=True)
        return sum(s) - sum(s[2::3])

print(Solution().minimumCost([1, 2, 3]), Solution().minimumCost([6, 5, 7, 9, 2, 2]), Solution().minimumCost([5, 5]))

5 23 10


## 10. Vì sao thuật toán đúng? (lập luận hoán đổi)

Mỗi "nhóm khuyến mãi" gồm 3 viên: hai viên trả tiền `x >= y` và một viên miễn phí `z <= y`. Ta muốn **tối đa tổng giá các viên miễn phí**.

- **Viên đắt nhất không bao giờ được miễn phí.** Viên miễn phí phải `<= giá viên rẻ hơn trong hai viên đã mua`, nên nó không thể lớn hơn giá một viên khác trong cùng nhóm; viên đắt nhất toàn bộ (hoặc bằng nhau thì có thể hoán đổi) luôn phải ở vị trí "trả tiền".
- **Hoán đổi:** giả sử một cách chia tối ưu có nhóm `(a, b, free)`, nhưng tồn tại viên `v` chưa nằm trong nhóm tốt nhất mà `v > free` và `v <= b`. Thay `free` bằng `v` sẽ không làm tăng chi phí (tổng tiền trả không tăng vì ta có thể hoán đổi `free` với `v`, giảm hoặc giữ nguyên chi phí). Do đó trong nghiệm tối ưu, viên miễn phí nên là viên **lớn nhất có thể** — chính là viên đắt thứ ba khi đã chọn hai viên đắt nhất.
- Lặp lại cho các viên còn lại: kết quả là sắp xếp giảm dần rồi chia nhóm 3, bỏ viên thứ 3 mỗi nhóm.
- **Hợp lệ:** `s[i+2] <= s[i+1]` nên điều kiện đề luôn thỏa.

Để chắc chắn, ta kiểm chứng bằng vét cạn nhỏ (mọi cách chia nhóm) ở dưới.

In [17]:
# Vét cạn: thử mọi cách chọn "tập viên miễn phí" hợp lệ cho n nhỏ rồi so với greedy
from itertools import permutations
import random

def exhaustive(cost):
    n = len(cost)
    best = sum(cost)
    for perm in permutations(range(n)):
        # đọc hoán vị theo nhóm 3: (mua a, mua b, tặng c) với c <= min(a, b)
        total, ok, i = 0, True, 0
        while i < n:
            grp = perm[i:i + 3]
            if len(grp) == 3:
                a, b, c = (cost[g] for g in grp)
                if c > min(a, b):
                    ok = False
                    break
                total += a + b
            else:
                total += sum(cost[g] for g in grp)
            i += 3
        if ok:
            best = min(best, total)
    return best

random.seed(2144)
for _ in range(200):
    c = [random.randint(1, 9) for _ in range(random.randint(1, 7))]
    assert exhaustive(c) == minimum_cost_slice(c), c
print("greedy khớp vét cạn trên 200 ca ngẫu nhiên")

greedy khớp vét cạn trên 200 ca ngẫu nhiên


## 11. Phân tích Time / Space Complexity

`n = len(cost) ≤ 100`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute mô phỏng | `O(n²)` | `O(n)` |
| 2. Sort + `i % 3` | `O(n log n)` | `O(n)` |
| 3. Sort + `s[2::3]` | `O(n log n)` | `O(n)` |
| 4. Heap | `O(n log n)` | `O(n)` |
| (Mở rộng) Counting sort | `O(n + V)` với `V = 100` | `O(V)` |

Thành phần chi phối là việc sắp xếp. Phần cộng dồn chỉ là `O(n)`.

## 12. Bộ test

Ví dụ đề, edge case (1 viên, 2 viên, 3 viên, mọi giá bằng nhau, số lượng không chia hết cho 3) và đối chiếu ngẫu nhiên cả 4 cách.

In [18]:
ALL = [minimum_cost_brute, minimum_cost_sort_loop, minimum_cost_slice, minimum_cost_heap]
cases = [
    ([1, 2, 3], 5),
    ([6, 5, 7, 9, 2, 2], 23),
    ([5, 5], 10),
    ([7], 7),
    ([1, 1, 1], 2),
    ([100, 100, 100, 100], 300),
    ([1, 2, 3, 4, 5], 12),         # sorted 5,4,3,2,1 -> trả 5+4+2+1 = 12
    ([3, 3, 3, 3, 3, 3], 12),
    ([10, 1, 1, 1], 12),           # 10,1 trả, 1 tặng, 1 trả = 12
]
for c, exp in cases:
    for f in ALL:
        assert f(c) == exp, (f.__name__, c, exp)
    assert Solution().minimumCost(c) == exp
print("Tất cả test ví dụ + edge case qua:", len(cases), "ca x", len(ALL) + 1, "cách")

Tất cả test ví dụ + edge case qua: 9 ca x 5 cách


In [19]:
import random
random.seed(21440)
for _ in range(3000):
    n = random.randint(1, 20)
    c = [random.randint(1, 100) for _ in range(n)]
    base = minimum_cost_brute(c)
    assert minimum_cost_sort_loop(c) == base
    assert minimum_cost_slice(c) == base
    assert minimum_cost_heap(c) == base
    assert Solution().minimumCost(c) == base
    assert c == c[:]        # đầu vào không bị sửa bởi lời giải nộp
print("3000 test ngẫu nhiên: các cách khớp nhau")

3000 test ngẫu nhiên: các cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Sắp xếp tăng dần rồi bỏ viên thứ ba

Sắp xếp tăng dần thì viên miễn phí lại là viên rẻ -> tiết kiệm được ít nhất.

In [20]:
def wrong_ascending(cost):
    s = sorted(cost)                      # SAI: tăng dần
    return sum(s) - sum(s[2::3])

print("sai:", wrong_ascending([6, 5, 7, 9, 2, 2]), "| đúng:", Solution().minimumCost([6, 5, 7, 9, 2, 2]))

sai: 17 | đúng: 23


Muốn **tặng viên đắt**, ta phải **nhóm các viên đắt lại với nhau** -> sắp xếp **giảm dần**. Sắp xếp tăng dần làm viên miễn phí là viên rẻ ở giữa mỗi nhóm.

### Lỗi 2: Dùng lát cắt `s[3::3]` hoặc `s[::3]` sai điểm bắt đầu

In [21]:
def wrong_slice_start(cost):
    s = sorted(cost, reverse=True)
    return sum(s) - sum(s[3::3])          # SAI: bỏ viên chỉ số 3, 6, ...

print("sai:", wrong_slice_start([1, 2, 3]), "| đúng:", Solution().minimumCost([1, 2, 3]))

sai: 6 | đúng: 5


Viên được tặng có chỉ số `2, 5, 8, ...` (**đếm từ 0**), tức `s[2::3]`, không phải `s[3::3]`. Nhóm đầu tiên `(s[0], s[1], s[2])` có viên thứ ba ở **chỉ số 2**.

### Lỗi 3: Tặng cả khi nhóm chưa đủ 3 viên

Nếu số viên không chia hết cho 3, nhóm cuối có 1–2 viên phải trả đủ tiền. Cách dùng `i % 3 == 2` tự xử lý đúng, nhưng viết `len(s) // 3` rồi trừ 3 viên đầu tiên là sai.

In [22]:
def wrong_cheapest_free(cost):
    s = sorted(cost, reverse=True)
    k = len(s) // 3                       # đúng số viên được tặng, nhưng chọn SAI viên (rẻ nhất)
    return sum(s) - sum(s[len(s) - k:] if k else [])

print("sai:", wrong_cheapest_free([6, 5, 7, 9, 2, 2]), "| đúng:", Solution().minimumCost([6, 5, 7, 9, 2, 2]))
print("sai:", wrong_cheapest_free([5, 5]), "| đúng:", Solution().minimumCost([5, 5]), "(trường hợp này trùng hợp đúng)")

sai: 27 | đúng: 23
sai: 10 | đúng: 10 (trường hợp này trùng hợp đúng)


### Lỗi 4 (thêm): Sửa trực tiếp `cost` bằng `.sort()`

`cost.sort(reverse=True)` làm thay đổi đầu vào. Với LeetCode thường không sao, nhưng nếu hàm khác còn dùng `cost` thì bất ngờ. Nên dùng `sorted(...)` để tạo bản mới khi không cần tiết kiệm bộ nhớ.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "tối thiểu chi phí / tối đa lợi ích" với quy tắc nhóm cố định -> nghĩ **sắp xếp + tham lam**.
- Sắp xếp **giảm dần** khi muốn các phần tử lớn đi cùng nhau; **tăng dần** khi muốn ghép nhỏ với lớn.
- Mẫu "cứ `k` phần tử thì bỏ/tặng phần tử thứ `k`" -> `i % k`, lát cắt `s[k-1::k]`.
- Khi giá trị bị chặn nhỏ (≤ 100), cân nhắc **counting sort**.
- Luôn tự kiểm chứng greedy bằng brute force nhỏ (như mục 10).

## 15. Biến thể của bài

1. **Mua `k` tặng 1** (thay vì mua 2 tặng 1): nhóm kích thước `k + 1`, viên thứ `k + 1` miễn phí.
2. **Tối đa tiền tiết kiệm** (tổng các viên miễn phí) thay vì chi phí.
3. Cho phép chọn **nhiều hơn** số viên tặng... (khi bỏ điều kiện giá).
4. Nhận **counting sort** vì `cost[i] ≤ 100`: `O(n + 100)`.
5. LeetCode 1710 (Maximum Units on a Truck), 561 (Array Partition) — cũng là sort + tham lam theo nhóm.

In [23]:
def minimum_cost_k(cost, k):
    """Mua k viên được tặng 1: nhóm kích thước k+1, viên cuối miễn phí."""
    s = sorted(cost, reverse=True)
    return sum(s) - sum(s[k::k + 1])

def minimum_cost_counting(cost):
    """Counting sort vì 1 <= cost[i] <= 100."""
    cnt = [0] * 101
    for c in cost:
        cnt[c] += 1
    total, k = 0, 0
    for v in range(100, 0, -1):
        for _ in range(cnt[v]):
            if k % 3 != 2:
                total += v
            k += 1
    return total

print(minimum_cost_k([6, 5, 7, 9, 2, 2], 2), minimum_cost_k([6, 5, 7, 9, 2, 2], 1))
print(minimum_cost_counting([6, 5, 7, 9, 2, 2]))

23 17
23


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute mô phỏng | Lặp lấy `max`, bỏ viên thứ 3 | `O(n²)` | `O(n)` | Hiểu đề, `n` nhỏ |
| 2. Sort + `i % 3` | Duyệt theo thứ tự giảm | `O(n log n)` | `O(n)` | Dễ đọc, dễ biến thể |
| 3. Sort + `s[2::3]` | Tổng - tổng miễn phí | `O(n log n)` | `O(n)` | **Nên dùng**, ngắn nhất |
| 4. Heap | Lấy lớn nhất từng viên | `O(n log n)` | `O(n)` | Dữ liệu đến dần |
| Counting sort | Đếm tần suất giá ≤ 100 | `O(n + V)` | `O(V)` | Giá bị chặn nhỏ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 — Đoán output:** `sorted([3, 1, 2], reverse=True)[2::3]` là gì?

**Bài 2 — Tính tay:** `cost = [4, 4, 4, 1, 1, 1, 9]` -> tổng chi phí tối thiểu?

**Bài 3 — Điền TODO:** hoàn thành `free_candies(cost)` trả về **list** các giá viên được tặng.

**Bài 4 — Tự viết lại:** viết `exercise_minimum_cost(cost)` bằng cách khác (ví dụ vòng lặp chỉ số `range(0, n, 3)`).

**Bài 5 — Mini challenge:** viết `max_saving(cost, k)` trả về tổng tiền **tiết kiệm tối đa** với khuyến mãi mua `k` tặng 1.

Các hàm chỉ định nghĩa (không gọi) để chạy toàn notebook không gây lỗi.

In [24]:
def free_candies(cost):
    # TODO: trả list các giá được tặng (đã sắp xếp giảm dần)
    pass

# assert free_candies([6, 5, 7, 9, 2, 2]) == [6, 2]

In [25]:
def exercise_minimum_cost(cost):
    # TODO: dùng range(0, n, 3) trên danh sách đã sắp xếp giảm dần
    pass

# assert exercise_minimum_cost([6, 5, 7, 9, 2, 2]) == 23

In [26]:
def max_saving(cost, k):
    # TODO: mua k viên tặng 1
    pass

# assert max_saving([6, 5, 7, 9, 2, 2], 2) == 8

## 18. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | `[3,2,1]` sau sort | Chỉ số `2` là `1` | `[1]` |
| 2 | Sort giảm: `9,4,4,4,1,1,1` | Nhóm: `(9,4,4) (4,1,1) (1)` | Tặng `4` và `1`, trả `9+4+4+1+1... ` -> tính lại bằng `sum - tặng` |
| 3 | Sort giảm dần | Lấy `s[2::3]` | `sorted(cost, reverse=True)[2::3]` |
| 4 | `for i in range(0, n, 3)` | Cộng `s[i] + s[i+1]` (nếu có) | Cộng thêm `s[i+2]` nếu không muốn tặng thì bỏ qua |
| 5 | Tiết kiệm = tổng viên miễn phí | Viên miễn phí là `s[k::k+1]` | `sum(sorted(cost, reverse=True)[k::k+1])` |

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `[1]`. **Bài 2:** sắp xếp giảm dần `[9,4,4,4,1,1,1]`; viên miễn phí ở chỉ số `2, 5` là `4` và `1`; tổng `24`, trừ `5` còn **19**.

In [27]:
assert sorted([3, 1, 2], reverse=True)[2::3] == [1]
assert Solution().minimumCost([4, 4, 4, 1, 1, 1, 9]) == 19

def free_candies_answer(cost):
    return sorted(cost, reverse=True)[2::3]

def exercise_minimum_cost_answer(cost):
    s = sorted(cost, reverse=True)
    total = 0
    for i in range(0, len(s), 3):
        total += s[i]
        if i + 1 < len(s):
            total += s[i + 1]
    return total

def max_saving_answer(cost, k):
    return sum(sorted(cost, reverse=True)[k::k + 1])

assert free_candies_answer([6, 5, 7, 9, 2, 2]) == [6, 2]
assert exercise_minimum_cost_answer([6, 5, 7, 9, 2, 2]) == 23
assert exercise_minimum_cost_answer([4, 4, 4, 1, 1, 1, 9]) == 19
assert max_saving_answer([6, 5, 7, 9, 2, 2], 2) == 8
assert max_saving_answer([6, 5, 7, 9, 2, 2], 1) == 7 + 5 + 2   # s=[9,7,6,5,2,2], k=1: s[1::2] = 7,5,2
print("Đáp án bài tập: tất cả assert đều qua")

Đáp án bài tập: tất cả assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template**

```python
class Solution:
    def minimumCost(self, cost):
        s = sorted(cost, reverse=True)
        return sum(s) - sum(s[2::3])
```

**Checklist**

- [ ] Sắp xếp **giảm dần** (không phải tăng dần).
- [ ] Viên miễn phí ở chỉ số `2, 5, 8, ...` (đếm từ 0): `s[2::3]`.
- [ ] Nhóm cuối có 1–2 viên: trả đủ tiền, không có quà.
- [ ] Đã so greedy với brute force trên input nhỏ ngẫu nhiên.
- [ ] Đã test `[1,2,3]`, `[6,5,7,9,2,2]`, `[5,5]` và `[x]` một viên.